# Data Cleaning: Non-Standard Values

In this notebook, we investigate and clean non-standard values in the ILINet dataset. The analysis focuses on identifying placeholder values, determining their meaning, and applying appropriate transformations to preserve data integrity before further preprocessing.

#### 1. Import Libraries

In [ ]:
# Import libraries
import warnings

import pandas as pd
import numpy as np

from src.data_loader import load_ilinet

warnings.filterwarnings("ignore")

#### 2. Display Settings

In [2]:
# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 120)

#### 3. Load Dataset

In [3]:
# Load dataset
df = load_ilinet()

#### 4. Investigating Non-Standard Values

Before cleaning the dataset, we investigate the non-standard `"X"` values present in several columns. Understanding what these values represent is essential before deciding how they should be handled.

In [4]:
# Count occurrences of "X" in each column
x_summary = (pd.DataFrame({
        "count_of_X": (df == "X").sum()
    })
    .query("count_of_X > 0")
    .sort_values("count_of_X", ascending=False)
)

display(x_summary)

,count_of_X
REGION,1499
AGE 25-64,872
AGE 25-49,532
AGE 50-64,532


In [5]:
# Sample records containing "X"
display(df.loc[
        (df == "X").any(axis=1),
        ["YEAR", "WEEK", "REGION", "AGE 25-49", "AGE 25-64", "AGE 50-64"]
    ].head(20)
)

,YEAR,WEEK,REGION,AGE 25-49,AGE 25-64,AGE 50-64
0,1997,40,X,X,157,X
1,1997,41,X,X,151,X
2,1997,42,X,X,153,X
3,1997,43,X,X,193,X
4,1997,44,X,X,162,X
5,1997,45,X,X,148,X
6,1997,46,X,X,240,X
7,1997,47,X,X,293,X
8,1997,48,X,X,206,X
9,1997,49,X,X,282,X


In [6]:
# Inspect the most frequent values in each age group column
for col in ["AGE 25-49", "AGE 25-64", "AGE 50-64"]:
    print(f"\n{col}")
    display(df[col]
        .value_counts(dropna=False)
        .head(10)
    )


AGE 25-49


AGE 25-49
X       532
0        95
1461      3
2677      2
2943      2
1235      2
1081      2
862       2
2511      2
4455      2
Name: count, dtype: int64


AGE 25-64


AGE 25-64
X      872
0       95
225      4
206      3
292      3
962      3
289      3
564      3
619      3
162      2
Name: count, dtype: int64


AGE 50-64


AGE 50-64
X       532
0        95
736       3
374       3
627       3
405       3
771       3
4591      3
1470      2
1049      2
Name: count, dtype: int64

#### 5. Handling Non-Standard Values

In [7]:
# Replace "X" with NaN in affected columns
columns_with_x = x_summary.index.tolist()

df[columns_with_x] = (
    df[columns_with_x]
    .replace("X", np.nan)
)

In [8]:
# Verify that all "X" values have been replaced
x_summary_after = (pd.DataFrame({
        "count_of_x": (df == "X").sum()
    })
    .query("count_of_x > 0")
)

display(x_summary_after)

if x_summary_after.empty:
    print("All 'X' values have been successfully replaced with NaN.")
else:
    print("Some 'X' values are still present in the dataset.")

,count_of_x


All 'X' values have been successfully replaced with NaN.


#### 6. Saving the Cleaned Dataset

In [9]:
# Save the cleaned dataset for subsequent preprocessing
df.to_csv("data/cleaned/ILINet_cleaned.csv", index=False)

### Findings

The investigation revealed the following:

- `REGION` contains only `"X"` because the dataset is reported at the national level.
- The age-group columns use `"X"` as a placeholder for values that were not reported under a particular reporting scheme.
- The value `0` represents a valid observation indicating zero reported influenza-like illness cases, rather than missing data.

These findings confirm that `"X"` should not be interpreted as zero and requires special handling during preprocessing.

## Summary

Based on the findings, all occurrences of `"X"` in the affected columns were replaced with `NaN` to distinguish unavailable values from legitimate zero counts. A verification step confirmed that the replacement was successfully completed.

The dataset is now free of non-standard placeholder values and is ready for the next stage of preprocessing and feature engineering.